# Lightweight supervised adaptation

Starting from the composed modules, additionally adapt the backbone on the (≤1M-token) target-domain train split, then evaluate on the same target-domain test set used for the zero-shot conditions.

The composed delta from the zero-shot condition (`ΔW = ΔW_genre + ΔW_language`) is merged
directly into the backbone's weights (`merge_and_unload()`), producing a single frozen "composed
backbone." A fresh, small LoRA adapter (same config as module training: r=8, α=16, dropout=0.05,
on q/k/v/out_proj) is then trained on top of that frozen composed backbone, using the target-domain
train split. This keeps both modules present in every forward pass, while "additionally
adapt" stays lightweight, a new small adapter, not full fine-tuning.

## Conditions

| name | base the new LoRA is trained on top of | purpose |
|---|---|---|
| `composed_adapted` | merged (genre + language) backbone |
| `backbone_adapted` | plain backbone, no modules | control: does pre-composing the modules help at all over adapting from scratch on the same scarce data? |

**Not included:** full fine-tuning (all backbone params, no LoRA) as a third baseline. XGLM's ~256k
vocabulary makes its tied embedding/output matrix a large fraction of its parameters, and
full-parameter Adam optimizer state for all 564M params is a real OOM risk on this machine's
unified 17GB memory (as already seen with eval-time logits OOM'ing at batch 64). 


## 0. Setup

In [ ]:
import gc
import time
from pathlib import Path

import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

MODEL_NAME = "facebook/xglm-564M"
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
RESULTS_DIR = Path("results")
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


GENRE_ADAPTER = MODELS_DIR / "genre_module_dgt"
LANGUAGE_ADAPTER = MODELS_DIR / "language_module_books_fi"


MAX_LENGTH = 128
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]

LEARNING_RATE = 2e-4
BATCH_SIZE = 4
MAX_STEPS = 600
WARMUP_STEPS = 30
LOGGING_STEPS = 50
EVAL_STEPS = 150
EVAL_SUBSET_SIZE = 200   # cap on the dev-set size used for periodic eval during training

EVAL_BATCH_SIZE = 16  # inference-only test-set eval (no gradients), can be larger than training batch

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. Target-domain data (train / val / test)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_LENGTH)

def load_tokenized_split(path, max_examples=None):
    ds = load_dataset("json", data_files=str(path), split="train")
    if max_examples is not None and len(ds) > max_examples:
        ds = ds.shuffle(seed=SEED).select(range(max_examples))
    return ds.map(tokenize, remove_columns=ds.column_names)

TARGET_DIR = DATA_DIR / "target_domain_fi_dgt"
# train: the <=1M-whitespace-token cap 
train_ds = load_tokenized_split(TARGET_DIR / "train.jsonl")
# dev: used only for periodic monitoring prints during training, not a reported metric
dev_ds = load_tokenized_split(TARGET_DIR / "val.jsonl", max_examples=EVAL_SUBSET_SIZE)
# test: the held-out set both conditions are ultimately scored on
test_ds = load_tokenized_split(TARGET_DIR / "test.jsonl")
print(f"train: {len(train_ds):,} | dev (subset): {len(dev_ds):,} | test: {len(test_ds):,}")

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
# shuffle=False: keeps test-sentence order identical to the zero shot condition testing so the per-example
# loss arrays saved by both notebooks are index-aligned for a later paired significance test
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator)


Map:   0%|          | 0/52509 [00:00<?, ? examples/s]


Map:   2%|▏         | 936/52509 [00:00<00:05, 9323.82 examples/s]


Map:   5%|▍         | 2384/52509 [00:00<00:05, 9538.56 examples/s]


Map:   7%|▋         | 3441/52509 [00:00<00:04, 9942.75 examples/s]


Map:   9%|▊         | 4486/52509 [00:00<00:04, 10120.69 examples/s]

...


Generating train split: 0 examples [00:00, ? examples/s]


Generating train split: 2630 examples [00:00, 823640.67 examples/s]


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Map: 100%|██████████| 200/200 [00:00<00:00, 6989.34 examples/s]

train: 52,509 | dev (subset): 200 | test: 2,618


## 2. Base-model helpers

`load_plain_backbone`: no modules; 
`load_merged_composed_backbone`: loads both adapters, combines them exactly as in the zero-shot
`composed` condition (`add_weighted_adapter`, linear/additive), then `merge_and_unload()` merges the
combined delta into the weights, returning a plain (no-adapter) model that a fresh LoRA can be
attached to.

In [ ]:
def load_plain_backbone():
    # no-modules control base: the fresh LoRA trained on top of this is `backbone_adapted`
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
    return model


def load_merged_composed_backbone():
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
    peft_model = PeftModel.from_pretrained(base, GENRE_ADAPTER, adapter_name="genre")
    peft_model.load_adapter(str(LANGUAGE_ADAPTER), adapter_name="language")
    peft_model.add_weighted_adapter(
        adapters=["genre", "language"], weights=[1.0, 1.0],
        adapter_name="composed", combination_type="linear",
    )
    peft_model.set_adapter("composed")
    merged = peft_model.merge_and_unload()   # merges the delta in; no adapter object survives
    return merged


def fresh_lora_on_top(base_model):
    lora_cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.to(DEVICE)
    return model


def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

## 3. Training helper (same pattern as `02_train_modules.ipynb`)

In [ ]:
def train_adapted_module(name, base_loader, output_dir):
    print(f"\n=== training {name} ===")
    base = base_loader()
    model = fresh_lora_on_top(base)
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=f"runs/{name}",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="no",   
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=dev_ds,
        data_collator=collator,
    )


    baseline = trainer.evaluate()
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    final = trainer.evaluate()

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)   
    tokenizer.save_pretrained(output_dir)
    print(f"saved adapter to {output_dir} ({elapsed/60:.1f} min)")

    result = {
        "module": name,
        "baseline_dev_loss": baseline["eval_loss"],
        "final_dev_loss": final["eval_loss"],
        "minutes": elapsed / 60,
    }
    free(model, trainer, base)
    return result

## 4. Train `composed_adapted`

In [ ]:
composed_adapted_result = train_adapted_module(
    name="composed_adapted",
    base_loader=load_merged_composed_backbone,
    output_dir=MODELS_DIR / "composed_adapted",
)
composed_adapted_result


=== training composed_adapted ===


trainable params: 1,572,864 || all params: 566,036,480 || trainable%: 0.2779


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Model Preparation Time
150,3.362000,3.084122,0.004400
300,3.313700,3.037285,0.004400
450,3.209200,3.019010,0.004400
600,3.220900,3.012710,0.004400


saved adapter to models/composed_adapted (10.1 min)


{'module': 'composed_adapted',
 'baseline_dev_loss': 3.7663190364837646,
 'final_dev_loss': 3.012709856033325,
 'minutes': 10.08240643342336}

## 5. Train `backbone_adapted` (no-modules control)

In [ ]:
# control condition: fresh LoRA trained on top of the PLAIN backbone: isolates whether pre-composing the modules helps at all
# over just adapting from scratch on the same scarce data
backbone_adapted_result = train_adapted_module(
    name="backbone_adapted",
    base_loader=load_plain_backbone,
    output_dir=MODELS_DIR / "backbone_adapted",
)
backbone_adapted_result


=== training backbone_adapted ===


trainable params: 1,572,864 || all params: 566,036,480 || trainable%: 0.2779


Step,Training Loss,Validation Loss,Model Preparation Time
150,3.487300,3.193851,0.005000
300,3.358600,3.084996,0.005000
450,3.225000,3.040220,0.005000
600,3.231500,3.025229,0.005000


saved adapter to models/backbone_adapted (10.1 min)


{'module': 'backbone_adapted',
 'baseline_dev_loss': 4.319767951965332,
 'final_dev_loss': 3.025228500366211,
 'minutes': 10.053775731722514}

## 6. Evaluate both on the target-domain test set

In [ ]:
@torch.no_grad()
def evaluate(model, loader, label=""):
    loss_fct = nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
    per_example_losses = []

    t0 = time.time()
    for i, batch in enumerate(loader):
        model_inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(DEVICE)
        logits = model(**model_inputs).logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
        ).view(shift_labels.size())

        mask = (shift_labels != -100).float()
        seq_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        per_example_losses.extend(seq_losses.float().cpu().tolist())

        if (i + 1) % 4 == 0 and DEVICE == "mps":
            torch.mps.empty_cache()
        if (i + 1) % 20 == 0:
            print(f"  [{label}] batch {i+1}/{len(loader)} ({time.time()-t0:.0f}s elapsed)", flush=True)

    per_example_losses = torch.tensor(per_example_losses)
    mean_loss = per_example_losses.mean().item()
    print(f"  [{label}] done in {time.time()-t0:.0f}s")
    return {
        "mean_loss": mean_loss,
        "ppl": float(torch.exp(torch.tensor(mean_loss))),
        "per_example_losses": per_example_losses,
    }

### 6a. `composed_adapted` on test set

In [ ]:
results = {}

base = load_merged_composed_backbone()
model = PeftModel.from_pretrained(base, str(MODELS_DIR / "composed_adapted"))
model.to(DEVICE)
model.eval()
results["composed_adapted"] = evaluate(model, test_loader, label="composed_adapted")
free(model, base)
print(f"mean_loss={results['composed_adapted']['mean_loss']:.4f}  ppl={results['composed_adapted']['ppl']:.2f}")

  [composed_adapted] batch 20/164 (58s elapsed)


  [composed_adapted] batch 40/164 (92s elapsed)


  [composed_adapted] batch 60/164 (128s elapsed)


  [composed_adapted] batch 80/164 (172s elapsed)


  [composed_adapted] batch 100/164 (217s elapsed)


  [composed_adapted] batch 120/164 (260s elapsed)


  [composed_adapted] batch 140/164 (304s elapsed)


  [composed_adapted] batch 160/164 (339s elapsed)


  [composed_adapted] done in 346s


mean_loss=3.0070  ppl=20.23


### 6b. `backbone_adapted` on test set

In [9]:
base = load_plain_backbone()
model = PeftModel.from_pretrained(base, str(MODELS_DIR / "backbone_adapted"))
model.to(DEVICE)
model.eval()
results["backbone_adapted"] = evaluate(model, test_loader, label="backbone_adapted")
free(model, base)
print(f"mean_loss={results['backbone_adapted']['mean_loss']:.4f}  ppl={results['backbone_adapted']['ppl']:.2f}")

  [backbone_adapted] batch 20/164 (34s elapsed)


  [backbone_adapted] batch 40/164 (72s elapsed)


  [backbone_adapted] batch 60/164 (111s elapsed)


  [backbone_adapted] batch 80/164 (151s elapsed)


  [backbone_adapted] batch 100/164 (191s elapsed)


  [backbone_adapted] batch 120/164 (230s elapsed)


  [backbone_adapted] batch 140/164 (267s elapsed)


  [backbone_adapted] batch 160/164 (300s elapsed)


  [backbone_adapted] done in 307s


mean_loss=2.9969  ppl=20.02


## 7. Combined summary (zero-shot + supervised-adapted)

In [ ]:
import numpy as np
import pandas as pd

zero_shot = np.load(RESULTS_DIR / "zero_shot_per_example_losses.npz")
zero_shot_means = {name: zero_shot[name].mean() for name in zero_shot.files}

rows = []
for name, loss in zero_shot_means.items():
    rows.append({"condition": name, "setting": "zero-shot", "mean_loss": loss, "ppl": float(np.exp(loss))})
for name, out in results.items():
    rows.append({"condition": name, "setting": "supervised-adapted", "mean_loss": out["mean_loss"], "ppl": out["ppl"]})

summary = pd.DataFrame(rows).sort_values(["setting", "ppl"])
summary

,condition,setting,mean_loss,ppl
5,backbone_adapted,supervised-adapted,2.996853,20.022425
4,composed_adapted,supervised-adapted,3.006988,20.226393
1,genre_only,zero-shot,3.467448,32.054825
0,backbone,zero-shot,3.518377,33.729645
2,language_only,zero-shot,3.768940,43.334126
3,composed,zero-shot,3.860293,47.479279


In [ ]:
np.savez(
    RESULTS_DIR / "supervised_per_example_losses.npz",
    **{name: out["per_example_losses"].numpy() for name, out in results.items()},
)
print("saved per-example losses to results/supervised_per_example_losses.npz")

saved per-example losses to results/supervised_per_example_losses.npz
